In [ ]:
import sys
sys.path.insert(0, '../..')

import textwrap
import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoTokenizer, VisionEncoderDecoderModel, AutoImageProcessor
from sartor.modules.generate import generate
from sartor.modules.utils import json2csv

In [ ]:
MODELS_PATH = '../../models/sartor-pretrained'
CAPS_DIR = '../../data/pretrain/dataset_rsicd.json'
IMGS_DIR = '../../data/pretrain/RSICD_images'

MAX_NEW_TOKENS = 64
NUM_BEAMS = 4
N_SAMPLES = 20
COLS = 4

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

tokenizer = AutoTokenizer.from_pretrained(MODELS_PATH)
model = VisionEncoderDecoderModel.from_pretrained(MODELS_PATH).to(device).eval()
processor = AutoImageProcessor.from_pretrained(MODELS_PATH)

df = json2csv(CAPS_DIR)
test_df = df.drop_duplicates(subset='Image Name').sample(n=N_SAMPLES, random_state=42)

In [ ]:
results = []
for _, row in test_df.iterrows():
    img_path = f"{IMGS_DIR}/{row['Image Name']}"
    pred = generate(
        model, processor, tokenizer, img_path, device,
        max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS,
    )
    results.append((img_path, row['Image Name'], row['Caption'], pred))
    print(f"[{row['Image Name']}]  PRED: {pred}")

In [ ]:
rows = (len(results) + COLS - 1) // COLS
fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 4, rows * 5))
axes = axes.flatten()

for idx, (img_path, name, gt, pred) in enumerate(results):
    ax = axes[idx]
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    ax.set_title(name, fontsize=9, fontweight='bold')
    caption = f"GT: {textwrap.fill(gt, 40)}\nPRED: {textwrap.fill(pred, 40)}"
    ax.set_xlabel(caption, fontsize=7, ha='left', x=0)
    ax.set_xticks([])
    ax.set_yticks([])

for idx in range(len(results), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()